# Map Viewer & Animation — Tsunami-HySEA Output (Standalone)

**HySEALab · Postprocessing Notebooks · EDANYA Research Group, Universidad de Málaga**
*Edited by José Manuel González Vida*

> 🧪 **BETA** — this notebook is under active development and will be updated
> shortly. If you find problems, please open an issue on GitHub.

A **self-contained** viewer for the variables produced by a Tsunami-HySEA
simulation output: all the logic lives inside the notebook (no external
helper modules needed).

---

### Requirements

**Python packages** (any recent version):

```bash
conda install -c conda-forge numpy xarray matplotlib ipywidgets netcdf4 pillow
# optional: pip install hdf5plugin h5netcdf   (extra NetCDF engines/filters)
```

**Input data:** any Tsunami-HySEA NetCDF output file (`.nc`), selected with the
graphical browser in the first code cell.

---

### What it does

1. **Graphical selector** to choose the output NetCDF file and the figures folder.
2. **2D map viewer**: variable, time step, colour scale
   (`auto` / `robust_2_98` / `symmetric` / `custom`), scope (`current` / `full`),
   downsampling and configurable cmap. If you select `arrival_times`, it draws
   **isochrones** (auto/manual levels).
3. **Export** the map to PNG.
4. **Animation** over a time range (GIF) with inline preview and saving.

Run the cells in order: **Configuration → Load file → Viewer → Animation**.


## 1. Configuration — visual file selection

Navigate to your data folder, select a `.nc` file and click **“✓ Use this file”**.
This defines `NC_FILE` and `SAVE_DIR`, used by the rest of the cells.


In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")  # avoids 'NetCDF: HDF error' on network drives
try:
    import hdf5plugin  # registers extra HDF5 filters (zstd/blosc/lz4...); harmless if missing
except Exception:
    pass

# === Configuration: visual file and folder selection ===
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.animation as manimation
import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage

plt.rcParams["figure.figsize"] = (11, 5)

# Initial browser folder (home, or auto-detected EDANYA root)
_root = Path.home() / "HySEALab" / "EDANYA"
if not _root.is_dir():
    for _c in [Path.cwd(), *Path.cwd().parents]:
        if (_c / "Grids").is_dir() and (_c / "Simulations").is_dir():
            _root = _c
            break
_start = _root if _root.is_dir() else Path.home()

NC_FILE = None
SAVE_DIR = Path("figures")
_W = widgets.Layout(width="760px")

w_cwd = widgets.Text(value=str(_start), description="Folder:", layout=_W)
w_go = widgets.Button(description="Go", button_style="info")
w_up = widgets.Button(description="Up", button_style="info")
w_home = widgets.Button(description="Home", button_style="info")
w_refresh = widgets.Button(description="Refresh", button_style="info")
w_list = widgets.Select(options=[], rows=15, layout=_W)
w_use = widgets.Button(description="✓ Use this file", button_style="success")
w_sel = widgets.Text(value="", description="File:", layout=_W, disabled=True)
w_savedir = widgets.Text(value=str(SAVE_DIR), description="Figures:", layout=_W)
w_status = widgets.HTML(value="")

_state = {"cwd": _start, "busy": False}

def _entries(cwd):
    items = []
    try:
        for p in sorted(cwd.iterdir(), key=lambda x: (not x.is_dir(), x.name.lower())):
            if p.is_dir():
                items.append((f"📁 {p.name}/", f"D::{p.name}"))
            elif p.suffix.lower() == ".nc":
                items.append((f"🧾 {p.name}", f"F::{p.name}"))
    except PermissionError:
        pass
    return items

def _refresh(*_):
    _state["busy"] = True
    w_cwd.value = str(_state["cwd"])
    opts = _entries(_state["cwd"])
    w_list.options = opts if opts else [("(no folders or .nc files)", "NONE")]
    _state["busy"] = False

def _sel_path():
    val = w_list.value
    if not val or val == "NONE":
        return None, None
    kind, name = val.split("::", 1)
    return kind, _state["cwd"] / name

def _on_pick(change):
    if change.get("name") != "value" or _state["busy"]:
        return
    kind, target = _sel_path()
    if kind == "D" and target is not None:
        _state["cwd"] = target
        _refresh()

def _go(_):
    p = Path(w_cwd.value.strip()).expanduser()
    if p.is_dir():
        _state["cwd"] = p
        _refresh()
    else:
        w_status.value = f"<span style='color:red'>Invalid folder: {p}</span>"

def _up(_):
    _state["cwd"] = _state["cwd"].parent
    _refresh()

def _home(_):
    _state["cwd"] = Path.home()
    _refresh()

def _use(_):
    global NC_FILE
    kind, target = _sel_path()
    if kind != "F" or target is None:
        w_status.value = "<span style='color:red'>Select a .nc file (not a folder).</span>"
        return
    NC_FILE = str(target)
    w_sel.value = NC_FILE
    w_status.value = f"<span style='color:green'><b>File:</b> {target.name}</span>"

def _on_savedir(change):
    if change.get("name") == "value":
        globals()["SAVE_DIR"] = Path(change["new"].strip() or "figures").expanduser()

w_list.observe(_on_pick, names="value")
w_go.on_click(_go); w_up.on_click(_up); w_home.on_click(_home); w_refresh.on_click(_refresh)
w_use.on_click(_use)
w_savedir.observe(_on_savedir, names="value")

_refresh()
display(widgets.VBox([
    widgets.HTML("<b>Navigate to your data folder, select a .nc file and click “Use this file”.</b>"),
    widgets.HBox([w_home, w_up, w_refresh]),
    widgets.HBox([w_cwd, w_go]),
    w_list,
    widgets.HBox([w_use]),
    w_sel,
    w_savedir,
    w_status,
]))


def open_nc(path):
    """Open a NetCDF file trying several engines (netcdf4 / h5netcdf / scipy).

    If all fail, shows the error of EACH engine plus a diagnostic hint.
    """
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"File does not exist: {p}")
    if p.stat().st_size == 0:
        raise OSError(f"The file is empty (0 bytes): {p}")

    with open(p, "rb") as _fh:
        magic = _fh.read(4)
    is_nc = magic[:3] == b"CDF" or magic == b"\x89HDF"

    errors = []
    for eng in ("netcdf4", "h5netcdf", "scipy"):
        try:
            return xr.open_dataset(p, engine=eng, decode_timedelta=False)
        except Exception as e:
            errors.append(f"  - {eng}: {type(e).__name__}: {e}")

    hint = ("The file looks like HDF5/NetCDF but no engine can read it: it is probably "
            "CORRUPT (incomplete copy or interrupted write). Copy it again from the "
            "source or regenerate it; check with  ncdump -h  in a terminal.") if is_nc else \
           ("The first bytes do NOT look like NetCDF/HDF5: check that this is the right file.")
    raise OSError(
        f"Could not open '{p}' with any engine.\n"
        f"First bytes = {magic!r}.\n"
        f"Errors per engine:\n" + "\n".join(errors) + f"\n\n{hint}"
    )


def save_figure(fig, name, dpi=150):
    d = Path(SAVE_DIR); d.mkdir(parents=True, exist_ok=True)
    path = d / name
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Figure saved: {path}")
    return path

## 2. Load the NetCDF file

Opens the selected file into `ds` and lists its variables.


In [ ]:
# === Load the file ===
if not NC_FILE or not Path(NC_FILE).is_file():
    raise RuntimeError("Select a .nc file in the Configuration cell "
                       "(button “Use this file”).")

ds = open_nc(NC_FILE)
print(ds)
print("\nVariables:")
for name, da in ds.data_vars.items():
    print(f" - {name:15s} dims={da.dims}, shape={da.shape}, units={da.attrs.get('units', '-')}")

## 3. 2D map viewer

Support utilities (axis selection, colour-scale limits, downsampling) and the
interactive viewer. Supported variables: 2D `(lat, lon)` or 3D `(time, lat, lon)`.


In [ ]:
# === Map utilities (self-contained) ===

def pick_xy_dims(da):
    lon_c = ["lon", "longitude", "grid_lon", "x"]
    lat_c = ["lat", "latitude", "grid_lat", "y"]
    x = next((d for d in da.dims if d in lon_c), da.dims[-1])
    y = next((d for d in da.dims if d in lat_c), da.dims[-2])
    return x, y


def _finite(a):
    a = np.asarray(a)
    return a[np.isfinite(a)]


def scale_limits(vals, mode="auto", vmin=None, vmax=None, qlow=2, qhigh=98):
    f = _finite(vals)
    if f.size == 0:
        return None, None
    if mode == "robust_2_98":
        return float(np.percentile(f, qlow)), float(np.percentile(f, qhigh))
    if mode == "symmetric":
        m = float(np.nanmax(np.abs(f)))
        return -m, m
    if mode == "custom":
        return vmin, vmax
    return float(f.min()), float(f.max())


def downsample_2d(da2d, max_points=700_000):
    if da2d.ndim != 2:
        return da2d, 1
    n = int(da2d.size)
    if n <= max_points:
        return da2d, 1
    factor = int(np.ceil(np.sqrt(n / max_points)))
    d0, d1 = da2d.dims
    return da2d.coarsen({d0: factor, d1: factor}, boundary="trim").mean(), factor


def elapsed_label(tvals, idx):
    arr = np.asarray(tvals)
    if np.issubdtype(arr.dtype, np.datetime64):
        s = (arr[idx] - arr[0]) / np.timedelta64(1, "s")
        return f"{float(s):.0f} s (from t0)"
    return f"{float(arr[idx]) - float(arr[0]):.6g} (from t0)"


# Plottable variables: 2D, or 3D with a 'time' dimension
MAP_VARS = [v for v in ds.data_vars
            if ds[v].ndim == 2 or (ds[v].ndim == 3 and "time" in ds[v].dims)]
print("Plottable variables:", MAP_VARS)

# --- Arrival times (isochrones) -----------------------------------------------
def _find_arrival_var():
    for name in ["arrival_times", "arrival_time", "min_arrival_time", "eta_arrival"]:
        if name in ds.data_vars:
            return name
    cands = [v for v in ds.data_vars if "arrival" in v.lower()]
    return cands[0] if cands else None

ARRIVAL_VAR = _find_arrival_var()
if ARRIVAL_VAR:
    print(f"Arrival-times variable detected: '{ARRIVAL_VAR}' "
          f"(select it in the viewer to see the isochrones).")


def _draw_arrival_isochrones(ax, var, cmap_name="RdYlBu", maxtime_min=None, step_min=None):
    """Draw arrival-time isochrones on `ax`. Automatic levels if None."""
    import copy as _copy
    da = ds[var]
    xn, yn = pick_xy_dims(da)
    lons = np.asarray(da[xn].values, dtype=float)
    lats = np.asarray(da[yn].values, dtype=float)
    arr = np.asarray(da.transpose(yn, xn).values, dtype=float)
    arr = np.where(arr <= -9000.0, np.nan, arr)            # fill value -> NaN

    units = (da.attrs.get("units", "") or "").lower()
    to_min = ("second" in units) or units in ("s", "sec", "secs")
    if not units and np.isfinite(arr).any() and np.nanmax(arr) > 1000:
        to_min = True                                       # heuristic: large values -> seconds
    arr_min = arr / 60.0 if to_min else arr

    max_time = float(np.nanmax(arr_min))
    step = step_min if step_min is not None else (5 if max_time <= 120 else 15 if max_time <= 300 else 30)
    top = maxtime_min if maxtime_min is not None else (np.ceil(max_time / step) * step + step)
    levels = np.arange(0, top, step)
    if levels.size < 2:
        levels = np.array([0.0, float(step)])

    cmap = _copy.copy(plt.get_cmap(cmap_name))             # .copy() does not exist in matplotlib < 3.4
    cmap.set_bad(color="white", alpha=0.0)                  # land / NaN transparent
    masked = np.ma.masked_invalid(arr_min)
    cfm = ax.contourf(lons, lats, masked, levels=levels, cmap=cmap, extend="max")
    cl = ax.contour(lons, lats, masked, levels=levels, colors="black", linewidths=0.6, alpha=0.4)
    for t in ax.clabel(cl, inline=True, fontsize=9, fmt="%1.0f min", colors="black"):
        t.set_fontweight("bold")
    ax.set_title(f"Arrival times (max: {max_time:.1f} min)")
    ax.set_xlabel("Longitude [\u00b0E]")
    ax.set_ylabel("Latitude [\u00b0N]")
    ax.grid(True, linestyle=":", alpha=0.3)
    ticks = levels if len(levels) < 15 else levels[::2]
    return cfm, "Arrival time (min)", ticks


def plot_arrival_times(maxtime_min=None, step_min=None, cmap_name="RdYlBu", var=None, save_as=None):
    """Script version (creates its own figure). Automatic levels if None; manual if given."""
    var = var or ARRIVAL_VAR
    if var is None:
        print("This file has no arrival-times variable (arrival_times).")
        return None
    fig, ax = plt.subplots(figsize=(12, 9))
    cfm, lbl, ticks = _draw_arrival_isochrones(ax, var, cmap_name, maxtime_min, step_min)
    cb = fig.colorbar(cfm, ax=ax, ticks=ticks)
    cb.set_label(lbl)
    plt.tight_layout()
    if save_as:
        save_figure(fig, save_as)
    plt.show()
    return fig

In [ ]:
# === Interactive map viewer (with isochrone mode for arrival_times) ===
w_mv_var = widgets.Dropdown(options=MAP_VARS,
                            value=("eta" if "eta" in MAP_VARS else (MAP_VARS[0] if MAP_VARS else None)),
                            description="Variable:")
w_mv_time = widgets.IntSlider(value=0, min=0, max=0, step=1, description="t index:")
w_mv_scale = widgets.Dropdown(options=["auto", "robust_2_98", "symmetric", "custom"],
                              value="robust_2_98", description="Scale:")
w_mv_scope = widgets.Dropdown(options=[("current frame", "current"), ("all times", "full")],
                              value="current", description="Scope:")
w_mv_vmin = widgets.FloatText(value=0.0, description="vmin:")
w_mv_vmax = widgets.FloatText(value=1.0, description="vmax:")
w_mv_cmap = widgets.Dropdown(options=["jet", "viridis", "RdBu_r", "coolwarm", "terrain", "turbo",
                                      "RdYlBu", "RdYlBu_r"],
                             value="jet", description="Cmap:")
w_mv_maxpts = widgets.IntSlider(value=700_000, min=50_000, max=3_000_000, step=50_000, description="Max pts:")
# Arrival-times-specific controls (only active when the variable is arrival)
w_mv_at_mode = widgets.Dropdown(options=[("auto", "auto"), ("manual", "manual")],
                                value="auto", description="Isochrones:")
w_mv_at_max = widgets.FloatText(value=40.0, description="max min:")
w_mv_at_step = widgets.FloatText(value=5.0, description="step min:")
w_mv_fname = widgets.Text(value="map_plot.png", description="File:")
w_mv_save = widgets.Button(description="Save PNG", button_style="success")
out_mv = widgets.Output()


def _mv_is_arrival(var):
    return var is not None and (var == ARRIVAL_VAR or "arrival" in var.lower())


def _mv_is_time(var):
    return var is not None and "time" in ds[var].dims


def _mv_update_controls(*_):
    var = w_mv_var.value
    arrival = _mv_is_arrival(var)

    # Regular map controls
    if _mv_is_time(var) and not arrival:
        w_mv_time.max = max(0, int(ds[var].sizes["time"]) - 1)
        w_mv_time.disabled = False
    else:
        w_mv_time.max = 0
        w_mv_time.value = 0
        w_mv_time.disabled = True
    for w in (w_mv_scale, w_mv_scope):
        w.disabled = arrival
    w_mv_vmin.disabled = arrival or w_mv_scale.value != "custom"
    w_mv_vmax.disabled = arrival or w_mv_scale.value != "custom"

    # Isochrone controls
    w_mv_at_mode.disabled = not arrival
    manual = arrival and w_mv_at_mode.value == "manual"
    w_mv_at_max.disabled = not manual
    w_mv_at_step.disabled = not manual

    # Suitable default cmap when an arrival variable is selected
    if arrival and w_mv_cmap.value not in ("RdYlBu", "RdYlBu_r"):
        w_mv_cmap.value = "RdYlBu"


def _mv_get_frame(var):
    da = ds[var]
    if _mv_is_time(var):
        return da.isel(time=int(w_mv_time.value))
    return da


def _mv_render(save=False):
    var = w_mv_var.value
    if var is None:
        print("No plottable variables.")
        return

    fig, ax = plt.subplots(figsize=(11, 6))

    if _mv_is_arrival(var):
        if w_mv_at_mode.value == "manual":
            mx, st = float(w_mv_at_max.value), float(w_mv_at_step.value)
        else:
            mx, st = None, None
        cfm, lbl, ticks = _draw_arrival_isochrones(ax, var, cmap_name=w_mv_cmap.value,
                                                   maxtime_min=mx, step_min=st)
        cb = fig.colorbar(cfm, ax=ax, ticks=ticks)
        cb.set_label(lbl)
    else:
        da2d = _mv_get_frame(var)
        da_show, factor = downsample_2d(da2d, max_points=int(w_mv_maxpts.value))
        xn, yn = pick_xy_dims(da_show)
        if w_mv_scope.value == "full" and _mv_is_time(var):
            vals_for_limits = ds[var].values
        else:
            vals_for_limits = da_show.values
        vmin, vmax = scale_limits(vals_for_limits, mode=w_mv_scale.value,
                                  vmin=float(w_mv_vmin.value), vmax=float(w_mv_vmax.value))
        arr = da_show.transpose(yn, xn).values
        mesh = ax.pcolormesh(np.asarray(da_show[xn].values), np.asarray(da_show[yn].values),
                             arr, cmap=w_mv_cmap.value, vmin=vmin, vmax=vmax, shading="auto")
        cb = fig.colorbar(mesh, ax=ax)
        cb.set_label(ds[var].attrs.get("units", "-"))
        title = f"{var}"
        if _mv_is_time(var):
            tlab = elapsed_label(ds[var]["time"].values, int(w_mv_time.value)) if "time" in ds[var].coords else f"t={w_mv_time.value}"
            title += f" | {tlab} | frame {int(w_mv_time.value) + 1}/{int(ds[var].sizes['time'])}"
        if factor > 1:
            title += f" | DS x{factor}"
        ax.set_title(title)
        ax.set_xlabel(xn)
        ax.set_ylabel(yn)

    plt.tight_layout()
    if save:
        save_figure(fig, w_mv_fname.value.strip() or "map_plot.png")
    plt.show()


def _mv_on_change(*_):
    _mv_update_controls()
    with out_mv:
        out_mv.clear_output(wait=True)
        _mv_render(save=False)


def _mv_on_save(_):
    with out_mv:
        _mv_render(save=True)


for w in (w_mv_var, w_mv_time, w_mv_scale, w_mv_scope, w_mv_vmin, w_mv_vmax,
          w_mv_cmap, w_mv_maxpts, w_mv_at_mode, w_mv_at_max, w_mv_at_step):
    w.observe(_mv_on_change, names="value")
w_mv_save.on_click(_mv_on_save)

_mv_update_controls()
display(widgets.VBox([
    widgets.HBox([w_mv_var, w_mv_time]),
    widgets.HBox([w_mv_scale, w_mv_scope, w_mv_cmap]),
    widgets.HBox([w_mv_vmin, w_mv_vmax, w_mv_maxpts]),
    widgets.HBox([w_mv_at_mode, w_mv_at_max, w_mv_at_step]),
    widgets.HBox([w_mv_fname, w_mv_save]),
    out_mv,
]))
_mv_on_change()

## 4. Animation over a time range (GIF)

Animates a time-dependent 2D variable between `t start` and `t end`. **Preview**
shows the animation inline; **Save GIF** writes it to `SAVE_DIR`.


In [ ]:
# === Animation over a time range ===
ANIM_VARS = [v for v in ds.data_vars if ds[v].ndim == 3 and "time" in ds[v].dims]

if not ANIM_VARS:
    print("No time-dependent 2D variables to animate.")
else:
    nt0 = int(ds[ANIM_VARS[0]].sizes["time"])
    a_var = widgets.Dropdown(options=ANIM_VARS,
                             value=("eta" if "eta" in ANIM_VARS else ANIM_VARS[0]),
                             description="Variable:")
    a_t0 = widgets.IntSlider(value=0, min=0, max=nt0 - 1, step=1, description="t start:")
    a_t1 = widgets.IntSlider(value=nt0 - 1, min=0, max=nt0 - 1, step=1, description="t end:")
    a_scale = widgets.Dropdown(options=["auto", "robust_2_98", "symmetric", "custom"],
                               value="robust_2_98", description="Scale:")
    a_vmin = widgets.FloatText(value=0.0, description="vmin:")
    a_vmax = widgets.FloatText(value=1.0, description="vmax:")
    a_cmap = widgets.Dropdown(options=["jet", "viridis", "RdBu_r", "coolwarm", "turbo"],
                              value="jet", description="Cmap:")
    a_delay = widgets.IntSlider(value=200, min=40, max=1200, step=10, description="Delay ms:")
    a_maxpts = widgets.IntSlider(value=700_000, min=50_000, max=2_000_000, step=50_000, description="Max pts:")
    a_name = widgets.Text(value="map_anim.gif", description="GIF:")
    a_preview = widgets.Button(description="Preview", button_style="info")
    a_save = widgets.Button(description="Save GIF", button_style="success")
    a_out = widgets.Output()

    def _make_anim():
        var = a_var.value
        da = ds[var]
        nt = int(da.sizes["time"])
        t0 = max(0, min(nt - 1, int(a_t0.value)))
        t1 = max(0, min(nt - 1, int(a_t1.value)))
        if t1 < t0:
            t0, t1 = t1, t0
        da_sel = da.isel(time=slice(t0, t1 + 1))
        _, factor = downsample_2d(da_sel.isel(time=0), max_points=int(a_maxpts.value))
        if factor > 1:
            xn, yn = pick_xy_dims(da_sel.isel(time=0))
            da_show = da_sel.coarsen({yn: factor, xn: factor}, boundary="trim").mean()
        else:
            da_show = da_sel
        xn, yn = pick_xy_dims(da_show.isel(time=0))

        # colour limits over the whole selected time range
        vmin, vmax = scale_limits(da_show.values, mode=a_scale.value,
                                  vmin=float(a_vmin.value), vmax=float(a_vmax.value))

        fig, ax = plt.subplots(figsize=(9, 5))
        f0 = da_show.isel(time=0).transpose(yn, xn)
        xv = np.asarray(f0[xn].values); yv = np.asarray(f0[yn].values)
        mesh = ax.pcolormesh(xv, yv, np.asarray(f0.values), cmap=a_cmap.value,
                             vmin=vmin, vmax=vmax, shading="auto")
        cb = fig.colorbar(mesh, ax=ax); cb.set_label(da.attrs.get("units", "-"))
        ax.set_xlabel(xn); ax.set_ylabel(yn)
        tvals = da_show["time"].values if "time" in da_show.coords else np.arange(da_show.sizes["time"])

        def _title(i):
            tl = elapsed_label(tvals, i) if "time" in da_show.coords else f"t={i}"
            return f"{var} | {tl} | frame {i + 1}/{int(da_show.sizes['time'])}"

        ax.set_title(_title(0))

        def _update(i):
            fr = da_show.isel(time=i).transpose(yn, xn)
            mesh.set_array(np.asarray(fr.values).ravel())
            ax.set_title(_title(i))
            return [mesh]

        ani = manimation.FuncAnimation(fig, _update, frames=int(da_show.sizes["time"]),
                                       interval=int(a_delay.value), blit=False, repeat=True)
        return fig, ani, t0, t1

    def _on_preview(_):
        with a_out:
            a_out.clear_output(wait=True)
            print("Generating animation...")
            try:
                fig, ani, _, _ = _make_anim()
                a_out.clear_output(wait=True)
                display(HTML(ani.to_jshtml()))
                plt.close(fig)
            except Exception as e:
                print(f"Error: {e}")

    def _on_save(_):
        with a_out:
            a_out.clear_output(wait=True)
            try:
                fig, ani, t0, t1 = _make_anim()
                d = Path(SAVE_DIR); d.mkdir(parents=True, exist_ok=True)
                out_file = d / (a_name.value.strip() or "map_anim.gif")
                fps = max(1, int(round(1000.0 / float(a_delay.value))))
                prog = widgets.IntProgress(value=0, min=0, max=t1 - t0 + 1, description="Saving:")
                display(prog)
                ani.save(out_file, writer=manimation.PillowWriter(fps=fps),
                         progress_callback=lambda i, n: setattr(prog, "value", i + 1))
                prog.bar_style = "success"
                print(f"GIF saved: {out_file}  (frames={t1 - t0 + 1}, fps={fps})")
                display(HTML(ani.to_jshtml()))
                plt.close(fig)
            except Exception as e:
                print(f"Error: {e}")

    a_preview.on_click(_on_preview)
    a_save.on_click(_on_save)
    display(widgets.VBox([
        widgets.HBox([a_var, a_cmap]),
        widgets.HBox([a_scale, a_vmin, a_vmax]),
        widgets.HBox([a_t0, a_t1]),
        widgets.HBox([a_delay, a_maxpts, a_name]),
        widgets.HBox([a_preview, a_save]),
        a_out,
    ]))

## Notes

- **Self-contained**: it does not import `ui_lib` or `notebook_ts_utils`; copy it anywhere.
- **Scale**: `symmetric` works well for `eta` (positive/negative); `robust_2_98` for
  maxima (`max_height`, `max_mom_flux`).
- **Scope**: `current frame` computes the colour limits from the displayed step;
  `all times` fixes them using the whole time range (useful to compare frames).
- **Downsampling**: lower “Max pts” if the grid is very large and rendering is slow.
- Figures (PNG) and animations (GIF) are saved to `SAVE_DIR` (`figures/` by default).
